# `figures_add.ipynb` — supplementary figures (Swap-meta ablation)

This notebook is a companion to `figures.ipynb` (the main paper figures).
It contains the bar-plot code for the **swap-meta ablation** added in the
revision in response to Reviewer #1.

The ablation inverts the meta / non-meta assignment of MetaTune:

* default MetaTune: the prompt embedding (`no_mask_embed`) is meta, LoRA + decoder are non-meta.
* swap-meta:        the prompt embedding is non-meta, LoRA + decoder are meta.

All Dice values were parsed directly from the test-set inference output of the
released checkpoints (Zenodo DOI 10.5281/zenodo.20517421, `ablations/swap_meta/`
for the inverted variant and `semantic_main/` for the MetaTune reference).
Each row of the data arrays is a single (dataset, *N*) configuration and
contains the three random seeds {42, 40, 22} in that column order, matching
the convention of `figures.ipynb`.

Running this notebook writes one SVG per dataset to `./result-figures/` with
the prefix `abla-swap-` so the new files do not collide with the existing
`abla1-` ... `abla5-` outputs.

In [2]:
import os
import matplotlib.pyplot as plt
from matplotlib import pyplot
import matplotlib
import numpy as np

plt.rc('font', family='Arial')
matplotlib.rcParams['svg.fonttype'] = 'none'

os.makedirs('./result-figures', exist_ok=True)

##### ablation: meta / non-meta swap

In [ ]:
# Swap-meta (prompt = non-meta, LoRA + decoder = meta).
# Each row: (dataset, N=4) at seeds {42, 40, 22}.
# Source: Zenodo deposit -> ablations/swap_meta/<dataset>4_swap_meta_img256_*_<NNNN>/best.pth,
# where NNNN / 10000 is the test-set Dice reported by inference.py.
data_swap = np.array([
    [0.8455, 0.8647, 0.8287],   # BCCD (blood cell)
    [0.6708, 0.6817, 0.6826],   # Osteosarcoma
    [0.4850, 0.5030, 0.4924],   # BT474 (breast cancer)
    [0.4363, 0.4089, 0.4387],   # Huh7 (liver cancer)
    [0.2659, 0.2992, 0.1746],   # Multi-modality
    [0.4445, 0.4536, 0.4662],   # CytoNuke (HNSCC)
])

# Default MetaTune (prompt = meta, LoRA + decoder = non-meta).
# Same values used as the MetaTune reference in every ablation cell of
# figures.ipynb, so this row stays consistent with the paper's main results.
data_meta = np.array([
    [0.8649, 0.8704, 0.8710],   # BCCD
    [0.7056, 0.6696, 0.7061],   # Osteo
    [0.5560, 0.5681, 0.5490],   # BT474
    [0.5685, 0.5750, 0.5644],   # Huh7
    [0.3978, 0.3749, 0.3620],   # Multi-modality
    [0.5004, 0.5047, 0.5128],   # CytoNuke
])

y1 = np.mean(data_swap, axis=1)
y2 = np.mean(data_meta, axis=1)
std_err1 = np.std(data_swap, axis=1)
std_err2 = np.std(data_meta, axis=1)

avg_list = [y1, y2]
mean_list = [std_err1, std_err2]

tick_label = ['Blood cell segmentation',
              'Osteosarcoma cell segmentation',
              'Breast cancer cell segmentation',
              'Liver cancer cell segmentation',
              'Multi-modality cell segmentation',
              'Head and neck squamous cell\ncarcinoma cell segmentation']
file_name  = ['abla-swap-bccd',  'abla-swap-osteo', 'abla-swap-bt474',
              'abla-swap-huh7',  'abla-swap-multimodal', 'abla-swap-cyto']

error_params = dict(elinewidth=1, ecolor='black', capsize=0)
bar_width = 0.2
colors = [
    (245/255., 198/255., 196/255.),
    (232/255., 192/255.,  71/255.),
    (150/255., 195/255., 110/255.),
]

for i in range(len(y1)):
    fig, ax = plt.subplots(figsize=(0.8, 2))
    plt.bar(0,         y1[i], bar_width, color=colors[0],
            yerr=std_err1[i], error_kw=error_params,
            label='Swap meta')
    plt.bar(bar_width, y2[i], bar_width, color=colors[1],
            yerr=std_err2[i], error_kw=error_params,
            label='MetaTune')
    for j in range(2):
        plt.text(bar_width*j-0.02,
                 avg_list[j][i] + mean_list[j][i] + 0.03,
                 f"{avg_list[j][i]:.2f}",
                 rotation=90., color='black', size=6)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    if i == 0:
        plt.ylabel('Dice', labelpad=-5, y=1.02, rotation=0, fontsize=6)
        plt.legend(loc="upper center", prop={'size': 6}, ncol=2,
                   frameon=False, bbox_to_anchor=(0.5, 1.35))
    else:
        plt.ylabel('Dice', labelpad=-5, y=1.02, rotation=0, fontsize=6)
    plt.title(f'{tick_label[i]}', size=6, x=0.5, y=1.05)

    ymin = round(np.min(np.array([y1[i], y2[i]])) - 0.1, 1)
    ymax = round(np.max(np.array([y1[i], y2[i]])) + 0.1, 1)
    # Per-task y-axis tweaks (matched to the other ablation cells in figures.ipynb)
    if   i == 0: ymin, ymax = 0.8, 0.91
    elif i == 1: ymin, ymax = 0.6, 0.71
    elif i == 2: ymin, ymax = 0.4, 0.61
    elif i == 3: ymin, ymax = 0.4, 0.61
    elif i == 4: ymin, ymax = 0.1, 0.41
    elif i == 5: ymin, ymax = 0.4, 0.51

    my_y_tick = np.arange(ymin, ymax, 0.1)
    plt.yticks(my_y_tick, size=6)
    plt.ylim(bottom=ymin)
    plt.xticks([])
    plt.savefig(f'./result-figures/{file_name[i]}.svg',
                format="svg", transparent=True,
                bbox_inches="tight", pad_inches=0.1)
    plt.close()

print('Wrote', [f'./result-figures/{n}.svg' for n in file_name])

##### comparison with additional few-shot baselines (HSNet, PerSAM-F, Matcher)

Per Reviewer #2's request that we benchmark MetaTune against state-of-the-art
few-shot segmentation baselines, we compare to **HSNet** (Min et al., *ICCV
2021*), **PerSAM-F** (Zhang et al., *ICLR 2024*) and **Matcher** (Liu et al.,
*ICLR 2024*) under the same N-shot supports and the same three random seeds
{42, 40, 22} as MetaTune. For brevity, this section uses the precomputed
mean ± std arrays directly (the per-seed Dice values are documented in
`HYPERPARAMETERS.md` and reproducible from the released checkpoints +
`config.txt` on Zenodo).

* **HSNet**     — eight datasets × three seeds (the same numbers shown in `figures.ipynb` cell 6).
* **PerSAM-F**  — eight datasets × three seeds (24 runs).
* **Matcher**   — seven datasets × three seeds (Matcher's official release does not include the GPU memory budget needed for Sartorius at 518×518; we leave that slot blank in the figure).
* **MetaTune**  — same eight datasets × three seeds, taken from the main-paper figure (`figures.ipynb` cell 2) so the bars are directly comparable to the main results.

In [9]:
# Precomputed mean ± std (over 3 seeds) per baseline, in the dataset order:
#   [BCCD, Osteo, BT474, Huh7, MultiModal, CytoNuke, FluoRed, Sartorius]
#
# Sources (per-seed values reproducible from the Zenodo deposit):
#   HSNet     -> figures.ipynb cell 6 (data1)
#   PerSAM-F  -> output_baselines/persam_f/<dataset><N>_persam_f_img256_seed<S>_<NNNN>
#   Matcher   -> output_baselines/matcher/<dataset><N>_matcher_img518_seed<S>_<NNNN>
#   MetaTune  -> figures.ipynb cell 2 (data4, first 8 rows)

y_hsnet   = np.array([0.590, 0.556, 0.430, 0.373, 0.248, 0.440, 0.340, 0.159])
s_hsnet   = np.array([0.020, 0.036, 0.009, 0.056, 0.010, 0.029, 0.039, 0.013])

y_persam  = np.array([0.753, 0.429, 0.456, 0.331, 0.320, 0.485, 0.172, 0.200])
s_persam  = np.array([0.051, 0.020, 0.023, 0.002, 0.056, 0.003, 0.024, 0.009])

# Matcher: no Sartorius run (518x518 input, GPU memory) -> NaN so its bar is skipped.
y_matcher = np.array([0.757, 0.601, 0.517, 0.529, 0.338, 0.410, 0.148, 0.251])
s_matcher = np.array([0.054, 0.028, 0.002, 0.024, 0.025, 0.001, 0.009, 0.015])

y_meta    = np.array([0.869, 0.694, 0.558, 0.569, 0.378, 0.506, 0.442, 0.395])
s_meta    = np.array([0.003, 0.021, 0.010, 0.005, 0.018, 0.006, 0.012, 0.007])

avg_list  = [y_persam,  y_matcher,  y_meta]
err_list  = [s_persam,  s_matcher,  s_meta]
labels    = ['PerSAM-F', 'Matcher',  'MetaTune']

tick_label = ['Blood cell segmentation',
              'Osteosarcoma cell segmentation',
              'Breast cancer cell segmentation',
              'Liver cancer cell segmentation',
              'Multi-modality cell segmentation',
              'Head and neck squamous cell\ncarcinoma cell segmentation',
              'Neuronal cell segmentation (FluoRed)',
              'Neuronal cell segmentation (Sartorius)']
file_name  = ['fewshot-bccd',  'fewshot-osteo', 'fewshot-bt474', 'fewshot-huh7',
              'fewshot-multimodal', 'fewshot-cyto', 'fewshot-fluored', 'fewshot-sartorius']

error_params = dict(elinewidth=1, ecolor='black', capsize=0)
bar_width = 0.2
colors = [
    (232/255., 192/255.,  71/255.),   # PerSAM-F  (yellow)
    (150/255., 195/255., 110/255.),   # Matcher   (green)
    (183/255., 142/255., 114/255.),   # MetaTune  (brown — matches figures.ipynb cell 2)
]

for i in range(len(y_meta)):
    fig, ax = plt.subplots(figsize=(1.5, 2))
    for j, (avg, err, lbl) in enumerate(zip(avg_list, err_list, labels)):
        v = avg[i]
        if np.isnan(v):
            # Zero-height stub keeps the legend entry; annotate slot with 'N/A'.
            plt.bar(bar_width * j, 0, bar_width, color=colors[j], label=lbl)
            plt.text(bar_width * j - 0.07, 0.005, 'N/A',
                     rotation=90., color='gray', size=6)
            continue
        plt.bar(bar_width * j, v, bar_width,
                color=colors[j], yerr=err[i],
                error_kw=error_params, label=lbl)
        plt.text(bar_width * j - 0.02,
                 v + err[i] + 0.03,
                 f"{v:.2f}",
                 rotation=90., color='black', size=6)

    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.ylabel('Dice', labelpad=-5, y=1.02, rotation=0, fontsize=6)
    if i == 0:
        plt.legend(loc="upper center", prop={'size': 6}, ncol=4,
                   frameon=False, bbox_to_anchor=(0.5, 1.35))
    plt.title(f'{tick_label[i]}', size=6, x=0.5, y=1.05)

    # Per-dataset y-axis tweaks, matched to the main-paper bar plots.
    if   i == 0: ymin, ymax = 0.6, 0.91
    elif i == 1: ymin, ymax = 0.3, 0.81
    elif i == 2: ymin, ymax = 0.4, 0.61
    elif i == 3: ymin, ymax = 0.3, 0.61
    elif i == 4: ymin, ymax = 0.2, 0.41
    elif i == 5: ymin, ymax = 0.4, 0.51
    elif i == 6: ymin, ymax = 0.1, 0.51
    elif i == 7: ymin, ymax = 0.1, 0.51

    plt.yticks(np.arange(ymin, ymax, 0.1), size=6)
    plt.ylim(bottom=ymin)
    plt.xticks([])
    plt.savefig(f'./result-figures/{file_name[i]}.svg',
                format="svg", transparent=True,
                bbox_inches="tight", pad_inches=0.1)
    plt.close()

print('Wrote', [f'./result-figures/{n}.svg' for n in file_name])

Wrote ['./result-figures/fewshot-bccd.svg', './result-figures/fewshot-osteo.svg', './result-figures/fewshot-bt474.svg', './result-figures/fewshot-huh7.svg', './result-figures/fewshot-multimodal.svg', './result-figures/fewshot-cyto.svg', './result-figures/fewshot-fluored.svg', './result-figures/fewshot-sartorius.svg']


##### instance-segmentation comparison — CytoNuke (N=4) & FluoRed (N=10)

Added in the revision in response to the reviewers' request that the paper
extend to instance segmentation. We compare four methods on the two datasets
for which instance-level ground truth is available:

* **StarDist** — radial-polygon instance head trained from scratch on the *N* supports.
* **Cellpose-SAM** — fine-tune the cpsam (Cellpose-SAM, ViT-L, 300 M params) backbone on the *N* supports.
* **Cellpose-Cyto** — fine-tune the legacy cyto3 backbone on the *N* supports.
* **Cellpose-SAM bilevel** — our cpsam fine-tune with the same bilevel meta / non-meta split as MetaTune.

The Cellpose evaluator (`metrics.average_precision`) reports AP at IoU
thresholds {0.5, 0.75, 0.9}. We report **AP@0.5**, **AP@0.75**, and an
approximate **mAP** defined as the mean of these three thresholds:

> **mAP ≈ (AP@0.5 + AP@0.75 + AP@0.9) / 3**

This is *not* the COCO-standard mAP@[0.5:0.95] in 0.05 steps; the caption
spells out the definition so readers know what is plotted. Bars show
**mean ± std over the three seeds {42, 40, 22}**, computed directly from the
per-seed arrays below (so the plotted values stay in sync with the data).

In [7]:
# Per-seed values for the four methods, 3 seeds each in order {42, 40, 22}.
# Each row of the arrays below is one method; columns are the three seeds.
# Source: result-figures/_instance_all_runs.csv (raw per-run rows).
methods_short = ['StarDist', 'Cellpose-SAM', 'Cellpose-Cyto', 'Cellpose-SAM bilevel']
seeds = [42, 40, 22]

# ---- CytoNuke (N=4) ----
data_ap50_cyto = np.array([
    [0.146, 0.112, 0.104],   # StarDist
    [0.424, 0.436, 0.373],   # Cellpose-SAM
    [0.425, 0.403, 0.332],   # Cellpose-Cyto
    [0.473, 0.396, 0.423],   # Cellpose-SAM bilevel
])
data_ap75_cyto = np.array([
    [0.008, 0.003, 0.003],
    [0.142, 0.168, 0.121],
    [0.123, 0.149, 0.136],
    [0.157, 0.216, 0.143],
])
data_mAP_cyto = np.array([
    [0.051, 0.039, 0.036],
    [0.187, 0.196, 0.169],   # mAP per seed = mean(AP@0.5, AP@0.75, AP@0.9)
    [0.164, 0.182, 0.129],
    [0.211, 0.205, 0.172],
])

# ---- FluoRed (N=10) ----
data_ap50_fluo = np.array([
    [0.173, 0.222, 0.217],
    [0.382, 0.409, 0.415],
    [0.369, 0.389, 0.407],
    [0.403, 0.426, 0.453],
])
data_ap75_fluo = np.array([
    [0.013, 0.021, 0.030],
    [0.122, 0.145, 0.176],
    [0.131, 0.120, 0.165],
    [0.153, 0.186, 0.204],
])
data_mAP_fluo = np.array([
    [0.062, 0.081, 0.082],
    [0.165, 0.186, 0.195],
    [0.159, 0.162, 0.179],
    [0.209, 0.298, 0.218],
])

# Same four-colour palette used elsewhere in the notebook (figures.ipynb cell 2):
# green, pink, yellow, brown.  Cellpose-SAM bilevel takes the brown slot so it
# keeps the same colour MetaTune has in the main-paper bar plots.
colors = [
    (150/255., 195/255., 110/255.),   # StarDist
    (245/255., 198/255., 196/255.),   # Cellpose-SAM
    (232/255., 192/255.,  71/255.),   # Cellpose-Cyto
    (183/255., 142/255., 114/255.),   # Cellpose-SAM bilevel
]

panels = [
    ('AP@0.5',  data_ap50_cyto, 'instance-cyto-ap50',    'CytoNuke / AP @ IoU 0.5'),
    ('AP@0.75', data_ap75_cyto, 'instance-cyto-ap75',    'CytoNuke / AP @ IoU 0.75'),
    ('mAP',     data_mAP_cyto,  'instance-cyto-mAP',     'CytoNuke / mAP*'),
    ('AP@0.5',  data_ap50_fluo, 'instance-fluored-ap50', 'FluoRed / AP @ IoU 0.5'),
    ('AP@0.75', data_ap75_fluo, 'instance-fluored-ap75', 'FluoRed / AP @ IoU 0.75'),
    ('mAP',     data_mAP_fluo,  'instance-fluored-mAP',  'FluoRed / mAP*'),
]

error_params = dict(elinewidth=1, ecolor='black', capsize=0)
bar_width = 0.2

for k, (metric, data, fname, title) in enumerate(panels):
    # Compute mean and std over the 3 seeds (axis=1) from the per-seed arrays.
    y = np.mean(data, axis=1)
    s = np.std(data,  axis=1)

    # Same figsize convention as the 4-bar few-shot figures.
    fig, ax = plt.subplots(figsize=(1.5, 2))
    for j in range(len(y)):
        plt.bar(bar_width * j, y[j], bar_width,
                color=colors[j],
                yerr=s[j],
                error_kw=error_params,
                label=methods_short[j])
        plt.text(bar_width * j - 0.02,
                 y[j] + s[j] + 0.01,
                 f"{y[j]:.2f}",
                 rotation=90., color='black', size=6)

    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    # plt.ylabel(metric, labelpad=-5, y=1.02, rotation=0, fontsize=6)
    if k == 0:   # legend only on the first SVG of the block
        plt.legend(loc="upper center", prop={'size': 6}, ncol=4,
                   frameon=False, bbox_to_anchor=(0.5, 1.35))
    plt.title(f'{title}', size=6, x=0.5, y=1.05)

    # Per-metric y-axis bounds (the three thresholds have very different scales).
    if   metric == 'AP@0.5':  ymin, ymax, step = 0.0, 0.51, 0.1
    elif metric == 'AP@0.75': ymin, ymax, step = 0.0, 0.26, 0.05
    else:                     ymin, ymax, step = 0.0, 0.31, 0.05

    plt.yticks(np.arange(ymin, ymax + 1e-6, step), size=6)
    plt.ylim(bottom=ymin, top=ymax)
    plt.xticks([])
    plt.savefig(f'./result-figures/{fname}.svg',
                format="svg", transparent=True,
                bbox_inches="tight", pad_inches=0.1)
    plt.close()

print('Wrote', [f'./result-figures/{p[2]}.svg' for p in panels])
print('*mAP definition: mean of (AP@0.5, AP@0.75, AP@0.9). Not COCO mAP@[0.5:0.95].')

Wrote ['./result-figures/instance-cyto-ap50.svg', './result-figures/instance-cyto-ap75.svg', './result-figures/instance-cyto-mAP.svg', './result-figures/instance-fluored-ap50.svg', './result-figures/instance-fluored-ap75.svg', './result-figures/instance-fluored-mAP.svg']
*mAP definition: mean of (AP@0.5, AP@0.75, AP@0.9). Not COCO mAP@[0.5:0.95].
